##### <font color="blue">*MCP (BigQuery) on Cloud Run with OIDC*

In [4]:
# @title Installs (Minimal für MCP & ADK)
############################################

# 1. Google Cloud ADK & GEAP
%pip install --quiet --upgrade "google-cloud-aiplatform[agent_engines,adk]" google-adk

# 2. MCP Client Library & Secret Manager (für Managed MCP)
%pip install --quiet mcp google-cloud-secret-manager

# 3. Version prüfen
%pip show google-adk mcp

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Name: google-adk
Version: 2.11.0
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: aiosqlite, authlib, click, fastapi, google-auth, google-genai, graphviz, httpx, jsonschema, opentelemetry-api, opentelemetry-sdk, packaging, pydantic, python-dotenv, python-multipart, pyyaml, requests, starlette, tenacity, typing-extensions, uvicorn, watchdog, websockets
Required-by: 
---
Name: mcp
Version: 1.30.0
Summary: Model Context Protocol SDK
Home-page: https://modelcontextprotocol.io
Author: Anthropic, PBC.
Author-email: 
License: MIT
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: anyio, httpx, httpx-sse, jsonschema, pydantic, pydantic-settings, pyjwt, python-multipart, sse-starlette, 

In [1]:
# @title Imports (Minimal für MCP & ADK)
############################################

import os
import asyncio
import warnings
warnings.filterwarnings('ignore')

# 1. Google Cloud Auth & Services
import google.auth
from google.auth.transport.requests import Request
from google.cloud import secretmanager
import vertexai

# 2. Google GenAI (Nachrichten-Typen)
from google.genai import types as genai_types

# 3. Google ADK (Agent, Runner & Sessions)
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, VertexAiSessionService

# 4. MCP Tools & Connection Types
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
from google.adk.tools.mcp_tool import SseConnectionParams
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

print("✅ Alle benötigten Imports erfolgreich geladen!")

✅ Alle benötigten Imports erfolgreich geladen!


In [2]:
# @title Environmental Variables
############################################

# These tell the underlying google-genai client to use Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = "deltorobarba"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"
# Required by LiteLLM for Vertex AI non-Gemini models
os.environ["VERTEXAI_PROJECT"] = "deltorobarba"
os.environ["VERTEXAI_LOCATION"] = "us-central1"
# Initialize Vertex AI client
client = vertexai.Client(project="deltorobarba", location="us-central1")
PROJECT_ID="deltorobarba"
LOCATION="us-central1"

In [3]:
# Test MCP connection
# https://console.cloud.google.com/run/detail/us-central1/bq-mcp-analyst/observability/metrics?project=deltorobarba
import asyncio
import google.auth
from google.auth.transport.requests import Request
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
from google.adk.tools.mcp_tool import SseConnectionParams

CLOUD_RUN_URL = "https://bq-mcp-analyst-622694106880.us-central1.run.app"

# 1. Lokale GCP-Credentials aus VS Code nutzen
creds, project_id = google.auth.default()
creds.refresh(Request())

# 2. SSE-Parameter mit OIDC ID-Token
bq_params = SseConnectionParams(
    url=f"{CLOUD_RUN_URL}/sse",
    headers={"Authorization": f"Bearer {creds.id_token}"}
)

# 3. Server anpingen & Tools abfragen
mcp_bq_tools = MCPToolset(connection_params=bq_params)
tools = await mcp_bq_tools.get_tools()
print("✅ Verbunden! Gefundene Tools:", [t.name for t in tools])

mTLS was requested but AsyncAuthorizedSession channel is not mTLS


✅ Verbunden! Gefundene Tools: ['query_public_dataset']


In [ ]:
# Test MCP server
# @title Connect to MCP on Cloud Run with OIDC and Request to BigQuery via MCP
############################################
import os
import google.auth
from google.auth.transport.requests import Request
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
from google.adk.tools.mcp_tool import SseConnectionParams
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

# Sicherstellen, dass Vertex AI für dein Projekt konfiguriert ist
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = "deltorobarba"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
# set "global" model for "gemini-3.8-flash"
# set "us-central1" for "gemini-3.5-flash"

"""
#The Cloud Run service was deployed with --no-allow-unauthenticated, so it requires an OIDC identity token (not an access token like OAuth2)
# The gcloud run services proxy is unreliable for SSE because it doesn't properly handle persistent long-lived connections.
# The fix: generate an OIDC identity token from the service account key and pass it directly as a header to SseConnectionParams.
# When running inm Colab: Get OIDC identity token (Cloud Run requires this, NOT an OAuth2 access token)
credentials = service_account.IDTokenCredentials.from_service_account_file(
    KEY_PATH,
    target_audience=CLOUD_RUN_URL  # <-- audience must match the Cloud Run URL exactly
)
credentials.refresh(google.auth.transport.requests.Request())
"""

CLOUD_RUN_URL = "https://bq-mcp-analyst-622694106880.us-central1.run.app"

# 1. Auth-Token über die lokalen VS Code GCP-Credentials beziehen
credentials, _ = google.auth.default()  # holt deine ADC-Identität
credentials.refresh(Request())          # füllt die Token-Felder
# Nach dem refresh() hängen an diesem einen Objekt zwei verschiedene Tokens (weiter unten in 2.):
# - credentials.token: OAuth2 Access-Token (ya29.…) für Google-APIs: Vertex AI, BigQuery, Discovery Engine
# - credentials.id_token: OIDC Identity-Token (JWT), beweist deine Identitaet.
    # Achtung: dieses User-Token hat als Audience die gcloud-Client-ID, nicht die
    # Service-URL. Cloud Run akzeptiert das fuer interaktive Nutzerkonten.
    # Ein DIENSTKONTO bekommt diese Kulanz nicht -> dort zwingend
    # google.oauth2.id_token.fetch_id_token(request, CLOUD_RUN_URL) mit exakter Audience.

# 2. Verbindungsparameter für Cloud Run (SSE) definieren (Pass token directly — no proxy needed)
bq_params = SseConnectionParams(
    url=f"{CLOUD_RUN_URL}/sse",
    # credentials.token vom Typ OAuth2 ACCESS-Token (ya29.…) für Google-APIs: Vertex AI, BigQuery, Discovery Engine
    # credentials.id_token vom Typ OIDC IDENTITY-Token (JWT: eyJhbGciOiJS…) prüfen deine Identität für Cloud Run, IAP (Audience muss exakt mit der URL übereinstimmen)
    # Cloud Run requires with '.id_token' an OIDC identity token (not an access token like OAuth2)
    headers={"Authorization": f"Bearer {credentials.id_token}"}
)

"""
Schritt 3 + 4 (nicht in diesem Notebook): die Server-Seite.

a) LAUFZEIT-IDENTITÄT — beim Deploy (Zelle 7).
   --service-account legt fest, als WER der Container innen arbeitet.
   Ab da liefert der Metadata-Server im Container diese Identität an ADC,
   d.h. bigquery.Client() in main.py ist automatisch dieses Dienstkonto.

   !gcloud run deploy {SERVICE_NAME} --source {DEPLOY_DIR}/ \
       --service-account="bq-mcp-analyst@deltorobarba.iam.gserviceaccount.com" \
       --no-allow-unauthenticated ...

b) RECHTE DES DIENSTKONTOS — einmalig per IAM.
   jobUser = darf im Projekt einen Query-Job starten (Kosten).
   Das Lesen von bigquery-public-data kommt vom Dataset selbst
   (fuer allAuthenticatedUsers freigegeben). Bei eigenen Tabellen
   zusaetzlich roles/bigquery.dataViewer.

   gcloud projects add-iam-policy-binding deltorobarba \
     --member="serviceAccount:bq-mcp-analyst@deltorobarba.iam.gserviceaccount.com" \
     --role="roles/bigquery.jobUser"

c) NICHT VERWECHSELN: roles/run.invoker gehoert dem AUFRUFER, nicht dem
   Dienstkonto. Steht in Zelle 7, Teil 2:
   gcloud run services add-iam-policy-binding bq-mcp-analyst \
     --member="user:alexmoheit@gmail.com" --role="roles/run.invoker"

# Die gesamte Kette
1. ADC lokal        google.auth.default()   -> wer bin ich?          (alexmoheit@gmail.com)
2. ID-Token daraus  credentials.id_token    -> Einlass bei Cloud Run (Aufrufer braucht run.invoker)
3. ADC im Container Metadata-Server         -> wer ist der Server?   (bq-mcp-analyst@…, via --service-account)
4. dessen Rolle     roles/bigquery.jobUser  -> darf er BigQuery?

401 = Token fehlt/falscher Typ (Access statt ID).
403 = Token ok, IAM-Rolle fehlt.
"""

# 1. Create the Toolset using your Cloud Run params
# (Make sure bq_params was defined in your previous step)
mcp_bq_tools = MCPToolset(connection_params=bq_params)
print("✅ Connected to Cloud Run MCP via OIDC token")

# 3. Agent definieren
enterprise_agent = LlmAgent(
    name='bigquery_analyst',
    model='gemini-3.8-flash',
    instruction="""
    You are an Enterprise Data Analyst with access to BigQuery.

    SQL RULES:
    - ALWAYS use SUM(number) and GROUP BY name to get totals.
    - ALWAYS wrap table names in backticks: `bigquery-public-data.usa_names.usa_1910_2013`.
    - Use the 'query_public_dataset' tool to get data.
    """,
    tools=[mcp_bq_tools]
)

# 4. Initialize Session Service & Runner
session_service = InMemorySessionService()
bq_runner = Runner(
    agent=enterprise_agent,
    session_service=session_service,
    app_name="cloud-mcp-demo"
)

# 5. Execute Query
question = "What were the top 3 most popular female names in the USA in 1980? Query the 'bigquery-public-data.usa_names.usa_1910_2013' table."
print(f"🤖 User: {question}\n")

async def run_analysis():
    # Create a fresh session
    sess = await session_service.create_session(app_name="cloud-mcp-demo", user_id="student-1")

    user_msg = genai_types.Content(
        role="user",
        parts=[genai_types.Part.from_text(text=question)]
    )

    async for event in bq_runner.run_async(
        user_id="student-1",
        session_id=sess.id,
        new_message=user_msg
    ):
        if event.is_final_response() and event.content:
            # print(f"🕵️ Analyst: {event.content.parts[0].text}") # before
            for part in event.content.parts:
                if hasattr(part, "text") and part.text:
                    print(f"🕵️ Analyst: {part.text}")

# Trigger the async function
await run_analysis()

✅ Connected to Cloud Run MCP via OIDC token
🤖 User: What were the top 3 most popular female names in the USA in 1980? Query the 'bigquery-public-data.usa_names.usa_1910_2013' table.



mTLS was requested but AsyncAuthorizedSession channel is not mTLS


🕵️ Analyst: The top 3 most popular female names in the USA in 1980 were:

1. **Jennifer** — 58,385 babies
2. **Amanda** — 35,818 babies
3. **Jessica** — 33,920 babies


In [ ]:
# @title Create files for Cloud Run Deployment
############################################

import os

DEPLOY_DIR = "cloud_mcp"
os.makedirs(DEPLOY_DIR, exist_ok=True)

# 1. Server Code mit FastMCP
server_code = """from mcp.server.fastmcp import FastMCP
from google.cloud import bigquery

# Initialize FastMCP
mcp = FastMCP("bigquery-analyst")

@mcp.tool()
def query_public_dataset(sql_query: str) -> str:
    """Run a read-only SQL query against BigQuery public datasets."""
    forbidden = ["DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "TRUNCATE"]
    if any(cmd in sql_query.upper() for cmd in forbidden):
        return "Error: Modification queries are not allowed. Read-only access."

    try:
        client = bigquery.Client()
        query_job = client.query(sql_query)
        # Direkt auf max 20 Zeilen begrenzen
        results = [dict(row) for row in query_job.result(max_results=20)]
        return str(results) if results else "No results found."
    except Exception as e:
        return f"BigQuery Error: {str(e)}"

# Starlette SSE-App exponieren
app = mcp.sse_app()
"""
with open(f"{DEPLOY_DIR}/main.py", "w") as f:
    f.write(server_code)

# 2. Requirements: mcp<2 pinnen (wichtig gegen MCP 2.x Breaking Changes!)
requirements_code = """mcp<2
google-cloud-bigquery
uvicorn
sse-starlette
"""
with open(f"{DEPLOY_DIR}/requirements.txt", "w") as f:
    f.write(requirements_code)

# 3. Procfile für Cloud Run Buildpacks
procfile_code = "web: uvicorn main:app --host 0.0.0.0 --port
"
with open(f"{DEPLOY_DIR}/Procfile", "w") as f:
    f.write(procfile_code)

print(f"✅ Deployment-Dateien erfolgreich in '{DEPLOY_DIR}/' bereitgestellt (inkl. Procfile & mcp<2 Fix)!")


In [ ]:
# @title Set Permissions and Deploy (⚠️ --> once only!)
############################################

PROJECT_ID = "deltorobarba"
REGION = "us-central1"
SERVICE_NAME = "bq-mcp-analyst"
DEPLOY_DIR = "cloud_mcp"
SERVICE_ACCOUNT = f"{SERVICE_NAME}@{PROJECT_ID}.iam.gserviceaccount.com"

# 1. Cloud Run Deployment direkt über deine aktive VS Code gcloud-Session
# (Kein Service-Account-Keyfile mehr nötig!)
!gcloud run deploy {SERVICE_NAME}     --source {DEPLOY_DIR}/     --platform managed     --region {REGION}     --service-account="{SERVICE_ACCOUNT}"     --no-allow-unauthenticated     --project={PROJECT_ID}     --quiet

# 2. Berechtigung: Sicherstellen, dass dein angemeldeter VS Code User den Dienst aufrufen darf
CURRENT_USER = !gcloud config get-value account
USER_EMAIL = CURRENT_USER[0].strip()

!gcloud run services add-iam-policy-binding {SERVICE_NAME}     --member="user:{USER_EMAIL}"     --role="roles/run.invoker"     --region={REGION}     --project={PROJECT_ID}     --quiet

print(f"✅ Deployment abgeschlossen! Invoker-Rechte für '{USER_EMAIL}' vergeben.")

In [ ]:
# @title Special: Managed MCP Server with Developer Knowledge API
############################################

import os
import asyncio
from google.cloud import secretmanager
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import VertexAiSessionService
from google.genai import types as genai_types

# ==========================================
# 1. CONFIGURATION
# ==========================================

# Vertex AI Backend konfigurieren
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = "deltorobarba"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

PROJECT_ID = "deltorobarba"
LOCATION = "us-central1"
SESSION_ID = "1152339862955753472"  # Vertex AI Agent Engine ID
SECRET_NAME = "mcp-dk"              # Developer Knowledge API Key im Secret Manager
DK_MCP_URL = "https://developerknowledge.googleapis.com/mcp"

# API-Key aus Google Cloud Secret Manager abrufen
def fetch_key():
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{PROJECT_ID}/secrets/{SECRET_NAME}/versions/latest"
    response = client.access_secret_version(request={"name": name})
    return response.payload.data.decode("UTF-8")

DK_API_KEY = fetch_key()

# ==========================================
# 2. TOOLSET INITIALIZATION
# ==========================================

dk_params = StreamableHTTPConnectionParams(
    url=DK_MCP_URL,
    headers={
        "X-Goog-Api-Key": DK_API_KEY,
        "Content-Type": "application/json"
    }
)

# McpToolset für den Managed MCP Server
google_docs_tools = McpToolset(connection_params=dk_params)

async def check_tools():
    try:
        tools = await google_docs_tools.get_tools()
        names = [t.name for t in tools]
        print(f"✅ Verbunden! Gefundene Tools: {names}")
    except Exception as e:
        print(f"❌ Connection Error: {e}")

# ==========================================
# 3. AGENT & RUNNER
# ==========================================

session_service = VertexAiSessionService(
    project=PROJECT_ID,
    location=LOCATION,
    agent_engine_id=SESSION_ID
)

docs_specialist = LlmAgent(
    name="google_docs_expert",
    model="gemini-3.8-flash",
    instruction="You are a Google Cloud expert. Use 'search_documents' or 'answer_query' to find info. List results in few bullet points and provide a code snippet.",
    tools=[google_docs_tools]
)

runner = Runner(
    agent=docs_specialist,
    session_service=session_service,
    app_name="dk_test"
)

async def ask_docs(question: str):
    # 1. Verbindung prüfen
    await check_tools()

    # 2. Session anlegen
    sess = await session_service.create_session(app_name="dk_test", user_id="student")
    user_msg = genai_types.Content(role="user", parts=[genai_types.Part.from_text(text=question)])

    print(f"🚀 Researching: {question}")

    # 3. Stream Response sauber ausgeben
    async for event in runner.run_async(user_id="student", session_id=sess.id, new_message=user_msg):
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if hasattr(part, "text") and part.text:
                    print(f"📖 RESPONSE:{part.text}")

# ==========================================
# 4. RUN
# ==========================================
await ask_docs("How does 'config=' need to look like if I want to deploy an agent with Agent Identity?")


mTLS was requested but AsyncAuthorizedSession channel is not mTLS


✅ Verbunden! Gefundene Tools: ['answer_query', 'get_documents', 'search_documents']
🚀 Researching: How does 'config=' need to look like if I want to deploy an agent with Agent Identity?
📖 RESPONSE:To deploy an agent with Agent Identity, the `config` parameter in the `client.agent_engines.create` method (for the Vertex AI Python SDK) needs to include the `identity_type` key set to `types.IdentityType.AGENT_IDENTITY`.

Here's how the `config` will look depending on your scenario:

*   **Deploying with Agent Code:**
    If you are deploying agent code (e.g., using `AdkApp`), the `config` should include the `identity_type` and necessary library requirements:

    ```python
    config={
        "identity_type": types.IdentityType.AGENT_IDENTITY,
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]",
            "google-adk[agent-identity]"
        ],
        # Optional fields supported by the SDK:
        "display_name": "your-agent-display-name",
        "stagi

---
## 🔬 Architekturvergleich: Dieses Notebook (`mcp.ipynb`) vs. `claude.ipynb`

### 1. Das mentale Modell & Gesamtüberblick
* **Dieses Notebook (`mcp.ipynb`):**
  Hier liegt der Fokus auf der **praktischen Server-Erstellung und dem isolierten Prototyping**. Du hast einen eigenen BigQuery-MCP-Server mit `FastMCP` geschrieben, Bereitstellungsdateien für Cloud Run erzeugt, den Dienst mit OIDC abgesichert und im Notebook per ADK `MCPToolset` sowie einen Managed MCP (Developer Knowledge API) direkt getestet.
* **Das Referenz-Notebook (`claude.ipynb`):**
  Claude baut den BigQuery-MCP-Server **nicht neu**, sondern greift auf deine existierende Cloud-Run-Instanz (`bq-mcp-analyst`) zu. Claudes Fokus liegt auf der **Enterprise-Härtung und Multi-Agent-Orchestrierung**: Wie bettet man diesen MCP-Server sauber in eine deploybare Vertex AI Reasoning Engine (Agent Runtime) ein, ohne dass Tokens ablaufen, Pickle-Fehler auftreten oder IAM-Berechtigungen fehlen?

---

### 2. Die 7 Kernunterschiede auf einen Blick

| Dimension | Deine Umsetzung (`mcp.ipynb`) | Claudes Umsetzung (`claude.ipynb`) | Warum der Unterschied wichtig ist |
|---|---|---|---|
| **1. Server-Rolle** | Du erstellst den MCP-Server (`cloud_mcp/main.py`) via `FastMCP` und deployst ihn auf Cloud Run. | Nutzt deinen Cloud-Run-Server wieder und ergänzt einen *externen* GitHub-MCP-Server. | `mcp.ipynb` liefert das Tool-Backend; `claude.ipynb` orchestriert mehrere MCPs im Agent-Team. |
| **2. Toolset-Initialisierung** | **Eager (Notebook-Zelle):**<br>`mcp_bq_tools = MCPToolset(...)` | **Lazy & serialisierungssicher:**<br>`LazyBigQueryMcpToolset(BaseToolset)` | `MCPToolset` hält Sockets/Locks. Beim Deploy auf Vertex AI bricht `pickle` ab oder überträgt veralteten State. |
| **3. Token-Lebensdauer** | **Statischer Header:**<br>`headers={"Authorization": f"Bearer {creds.id_token}"}` | **Dynamischer `header_provider`:**<br>`lambda ctx: {"Authorization": f"Bearer {cloud_run_id_token(url)}"}` | Google OIDC-Tokens laufen nach **60 min** ab. Nach 1 h wirft dein Agent `401 Unauthorized`. |
| **4. Identität & Audience** | Lokales ADC-Token des Nutzers (Audience = `gcloud`-Client-ID). | **Dual-Identity (`cloud_run_id_token`):** Deployed via Metadata-Server (`aud = CLOUD_RUN_URL`), lokal Fallback. | Cloud Run toleriert User-Tokens, verlangt bei Dienstkonten/Service Agents aber zwingend die exakte Service-URL als Audience. |
| **5. IAM-Rechte** | `roles/run.invoker` nur für deinen interaktiven User `user:{USER_EMAIL}`. | `roles/run.invoker` zusätzlich für den **Vertex AI Reasoning Engine Service Agent**. | Nach dem Deploy läuft der Agent als Service Agent; ohne diese Rolle scheitert der MCP-Call mit `403 Forbidden`. |
| **6. Externe MCPs / Auth** | API-Key aus Secret Manager statisch im Header übergeben. | **GCP Agent Identity (SPIFFE) + Auth Manager:** Dynamische Token-Auflösung, `tool_filter`. | Keine Keys im Code/State; feingranulares Least-Privilege-Prinzip auf Tool-Ebene. |
| **7. Governance & Registry** | Stand-alone Cloud Run Service ohne Katalogisierung. | Registrierung in der **GCP Agent Registry** (`gcloud alpha agent-registry services create`). | Sichtbarkeit in Agent Topology und Auditierung im Security Command Center (SCC AI Protection). |

---

### 3. Die technischen Details im Deep-Dive

#### A. Toolset-Lifecycle: Eager vs. Lazy Loading (Pickle-Sicherheit)
* In `mcp.ipynb` wird das `MCPToolset` direkt in Zelle 4/5 instanziiert. Für interaktive Notebook-Runs funktioniert das problemlos.
* Wenn ein Agent jedoch auf die **Vertex AI Agent Runtime (Reasoning Engines)** deployt wird (`client.agent_engines.create`), wird das gesamte Agent-Objekt mittels Pythons `pickle` serialisiert und in die Cloud übertragen.
* Ein aktives `MCPToolset` enthält offene HTTP-/SSE-Clients, Sockets, Locks und Event-Loops, die **nicht serialisierbar** sind.
* **Claudes Lösung (`LazyBigQueryMcpToolset` in `repo_intel/tools.py`):**
```python
class LazyBigQueryMcpToolset(BaseToolset):
    def __init__(self, cloud_run_url: str):
        super().__init__()
        self._url = cloud_run_url.rstrip("/")
        self._inner = None

    def _get_inner(self):
        if self._inner is None:
            self._inner = McpToolset(
                connection_params=SseConnectionParams(url=f"{self._url}/sse"),
                header_provider=lambda ctx: {"Authorization": f"Bearer {cloud_run_id_token(self._url)}"},
            )
        return self._inner

    def __getstate__(self):
        # Nur die String-URL wird gepickelt – keine Netzwerk-Sockets oder Sessions!
        return {"_url": self._url}
```
  Das Toolset wird erst **zur Laufzeit auf dem Server beim ersten Toolaufruf** erzeugt.

#### B. Token-Management: Statischer Header vs. Dynamischer Provider
* In `mcp.ipynb` liest du das Token einmalig aus:
```python
creds, _ = google.auth.default()
creds.refresh(Request())
bq_params = SseConnectionParams(
    url=f"{CLOUD_RUN_URL}/sse",
    headers={"Authorization": f"Bearer {creds.id_token}"}
)
```
  Dieses Token ist ein statischer Snapshot. Nach genau 3600 Sekunden (1 Stunde) wird jeder nachfolgende Aufruf von Cloud Run mit `401 Unauthorized` abgewiesen.
* In `claude.ipynb` wird stattdessen ein `header_provider` übergeben, der vor jedem Tool-Call evaluiert wird:
  - **Deployed:** `google.oauth2.id_token.fetch_id_token(request, audience)` fordert vom GCP Metadata-Server ein frisches Token an.
  - **Caching:** Ein In-Memory-Cache hält das Token für 55 Minuten und erneuert es automatisch rechtzeitig vor Ablauf.
  - **Lokal:** Automatischer Fallback auf `creds.id_token`.

#### C. Wer ruft auf? (Identitätskette & 401/403)
Beim Aufruf eines privaten Cloud Run MCP-Servers durch einen deployten Agenten gibt es vier Stationen:
1. **Lokal (dein Notebook):** Du (`user:{USER_EMAIL}`) rufst Cloud Run auf → Du hast `roles/run.invoker` → ✅ Funktioniert.
2. **Deployed (Agent Runtime):** Der Nutzer ruft die Root-Pipeline auf → Die Pipeline läuft als **Reasoning Engine Service Agent** (`service-{PROJECT_NUMBER}@gcp-sa-aiplatform-re...`) → Der Service Agent ruft Cloud Run auf → Er braucht zwingend `roles/run.invoker` auf dem Cloud-Run-Dienst!
3. **Im MCP-Container:** Der Cloud-Run-Dienst läuft als eigenes Dienstkonto (`bq-mcp-analyst@...`) → Dieses Dienstkonto benötigt `roles/bigquery.jobUser` im Projekt, um SQL-Abfragen auszuführen.

#### D. Externe MCP-Server: Secret Manager vs. SPIFFE / Auth Manager
* In Zelle 8 von `mcp.ipynb` wird für die Developer Knowledge API ein API-Key per SDK-Funktion aus dem Secret Manager geladen und als Header mitgegeben.
* In `claude.ipynb` wird der GitHub MCP Server über das Google Agent Identity Framework angebunden:
  - Der Agent erhält eine **SPIFFE-Identität** (`identity_type=AGENT_IDENTITY`).
  - Der GitHub-PAT wird im **GCP Auth Manager** (`github-mcp-auth-provider`) hinterlegt.
  - Das ADK löst das Credential über `GcpAuthProvider` erst just-in-time auf.
  - `tool_filter=["search_issues", "list_issues", ...]` stellt sicher, dass der Agent nur unkritische Lese-Tools ausführen darf (Prompt-Injection-Schutz).

#### E. Code-Organisation & Pickle-Hygiene
* In `mcp.ipynb` existiert der Client-Code in flachen Notebook-Zellen.
* In `claude.ipynb` wird der Code vor dem Deployment in die Python-Pakete `repo_intel/` und `github_agent/` geschrieben.
* **Hintergrund:** Würde man einen HTTP-Client oder Auth-Handler mit `google.auth.default()` direkt in einer Notebook-Zelle erzeugen und deployen, würde Python dessen internen Zustand samt deinem **persönlichen Refresh-Token** in das Deployment pickeln! Der Cloud-Agent würde dann mit deinen privaten Rechten statt mit der Service-Agent-Identität laufen. Die Module in `claude.ipynb` nutzen `__reduce__`, um beim Unpickling in der Cloud immer eine frische Instanz zu erzwingen.
